Stimulating Bessel's correction:
Recall that the sample variance is defined as above, with (N-1) in the denominator instead of (N) as one might
expect (and as seen in the population variance). Replacing (N) with (N -1) is known as Bessel’s Correction. It
makes the sample variance unbiased (the sample standard deviation is still slightly biased). In this homework,
you will simulate some confidence intervals to see how Bessel’s Correction affects the coverage.

# Part 1

In [28]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

### 1. Draw 20 random samples from a Normal distribution


In [3]:
# 1. Draw 20 random samples from a Normal distribution
N_SAMPLES = 20
normal_samples = np.random.normal(loc=0, scale=1, size=N_SAMPLES)

### 2. Compute sample mean

In [4]:
# 2. Compute sample mean
sample_mean = np.mean(normal_samples)
print(sample_mean)

-0.04897201900409626


### 3. Compute both the corrected and the uncorrected sample standard deviation


In [5]:
# 3. Compute both the corrected and the uncorrected sample standard deviation
uncorrected_std = np.std(normal_samples, ddof = 0)
corrected_std = np.std(normal_samples, ddof = 1)
print(uncorrected_std)
print(corrected_std)

0.8588147432192489
0.8811253349945197


### 4. Compute the critical value for a 95% confidence interval on the sample mean. State whether you used z or t, and why.

Since 20 samples are drawn from a Standard Normal distribution, we know that population standard deviation $\sigma = 1$. However, since both critical values are estimated from the sample mean $s$, using critical value z will add uncertainty. Therefore, I will use the critical value t. The t-distribution has heavier tails than z-distribution, and so can account for a larger uncertainty.

In [6]:
# 4. Compute the critical value for a 95% confidence interval on the sample mean.
# State whether you used z or t, and why.


# For a confidence level of 95%, a = 0.05
ALPHA = 0.05
critical_value_z = scipy.stats.norm.ppf(1 - ALPHA / 2) #account for both tails


df = N_SAMPLES - 1
critical_value_t = scipy.stats.t.ppf(1 - ALPHA / 2, df)
print(critical_value_z, critical_value_t)

1.959963984540054 2.093024054408263


### 5. Construct the 95% confidence intervals on the sample mean, using both the corrected and uncorrected sample standard deviations. Do these confidence intervals overlap with the population mean?

Since the 20 samples is drawn from a Standard Normal distribution, the population mean $\mu = 0$ and population standard deviation $ \sigma = 1$. So the intervals contain the true $\mu = 0$. Neither is centered on 0, because $\bar{x} = −0.049$ is a random draw. But the interval’s width is large enough to cover the gap.




In [7]:
# 5. Construct the 95% confidence intervals on the sample mean, using both the corrected and uncorrected sample standard deviations

# CI = Point Estimate (mean) + Margin of Error
# Margin of Error = Critical Value * Standard Error
# Standard Error = sigma/sqrt(N_SAMPLES)
# Use critical value t

m_error_corrected = critical_value_t * (corrected_std / np.sqrt(N_SAMPLES))
m_error_uncorrected = critical_value_t * (uncorrected_std / np.sqrt(N_SAMPLES))

print(m_error_corrected, m_error_uncorrected)

for label, sd in [("corrected", corrected_std), ("uncorrected", uncorrected_std)]:
    me = critical_value_t * sd / np.sqrt(N_SAMPLES)
    print(f"{label}: ({sample_mean - me:.4f}, {sample_mean + me:.4f})")

0.41237935063900394 0.40193767227242366
corrected: (-0.4614, 0.3634)
uncorrected: (-0.4509, 0.3530)


### 7. Repeat this exercise 10,000 times

In [8]:
# 7. Repeat this exercise 10,000 times
n_reps = 10000
n = 20
hits_corr = 0
hits_uncorr = 0

for _ in range(n_reps):
    sample = np.random.normal(0, 1, n)
    xbar = np.mean(sample)
    std_corr = np.std(sample, ddof=1)
    std_uncorr = np.std(sample, ddof=0)
    me_corr = critical_value_t * std_corr / np.sqrt(n)
    me_uncorr = critical_value_t * std_uncorr / np.sqrt(n)
    if xbar - me_corr < 0 and xbar + me_corr > 0:
        hits_corr += 1
    if xbar - me_uncorr < 0 and xbar + me_uncorr > 0:
        hits_uncorr += 1

print(f"Corrected CI contains the population mean in {hits_corr} of {n_reps} repetitions")
print(f"Uncorrected CI contains the population mean in {hits_uncorr} of {n_reps} repetitions")

Corrected CI contains the population mean in 9493 of 10000 repetitions
Uncorrected CI contains the population mean in 9438 of 10000 repetitions


### 8. How many times does the corrected confidence interval overlap with the population mean? How many times does the uncorrected confidence interval overlap with the population mean?

About 9493 times out of 10,000 repetitions have the corrected confidence interval overlap with the population mean. About 9438 times out of 10,0000 repetitions have the uncorrected confidence interval overlap with the population mean.

### 9. How many times should a properly calibrated confidence interval overlap with the population mean?
A properly calibrated confidence interval should overlap the population mean 9500 times.

### 10. Which interval is better calibrated?

A properly calibrated 95% interval should contain the population mean in about 95% of repetitions, so about 9,500 of 10,000. Even a perfectly calibrated procedure won’t hit 9,500 exactly. The count has a standard deviation of $\sqrt(10,000 × 0.95 × 0.05) ≈ 22$, so anything within about $9500 ±44$  (two SDs) is consistent with 95%.

The corrected confidence interval overlap the true population mean more (9,493 is within ±44 of 9,500, while 9,438 is 62 below, outside that band), and thus the corrected interval using the Bessel's correction is better.

# Part 2

### 1. The standard error of an estimated coverage fraction ĉ after N independent trials is √(c(1 − c)/N). At N = 1,000,000 and c ≈ 0.95, what is the standard error of your estimate?

The standard error is $≈0.000218$


In [9]:
# 1. The standard error of an estimated coverage fraction ĉ after N independent trials is √(c(1 − c)/N).
# At N = 1,000,000 and c ≈ 0.95, what is the standard error of your estimate?

n1 = 1000000
c = 0.95
se = np.sqrt((c * (1 - c)) / n1)
print(se)

0.00021794494717703377


### 2. Roughly how many trials do you need to distinguish a true coverage of 0.950 from 0.945 with about 95% confidence?

The difference between 0.95 and 0.945 is 0.005, so the estimate uncertainty must be smaller than that.
With 95% confidence, the $\alpha = 0.05$ and thus critical value z is $1.96$.

With $c \approx 0.95$, the margin of error is $1.96 \times \sqrt((0.95 * (1 - 0.95)) / n) = 1.96 \times \sqrt(0.0475/n) = 0.005 $. $$n = (0.0475 \times 1.96^2)/0.005^2 = 7299$$ At N ≈ 7,300, the margin of error equals the gap. If the true coverage is 0.945, the estimate lands on either side of 0.945 with equal chance. Only when it lands below does its 95% interval exclude 0.950, so at ≈ 7,300 trials, we would detect the difference about 50% of the time.

Another way to solve this problem is, we have to determine confidence intervals of coverage at 0.950 and 0.945, such that these 2 intervals don't overlap. This means the sum of the 2 margin of errors must be $<=$ 0.005, and each margin of error is $<= 0.0025$.$$1.96 \times \sqrt(\frac{0.95 \times (1-0.95)}{N}) = 0.0025$$
$$N = \frac{(0.0475 × 1.96²)}{0.0025^2}$$
$$N ≈29196$$

This is a conservative approach, such that at ≈ 29,000 trials, the two coverage intervals would be non-overlapping.

In [18]:
# Minimum trials needed
n_val = (0.0475 * (1.96**2)) / (0.005**2)
print(f"n = {n_val:.2f}")

n = 7299.04


In [19]:
# Stricter criteria: 2 intervals don't overlap
n_val = (0.0475 * (1.96**2)) / (0.0025**2)
print(f"n = {n_val:.2f}")

n = 29196.16


### Why is it better to evaluate both intervals on the same simulated samples than on separate ones?

The two intervals share the same $\bar{x}$ and only differ by the width of the estimated standard deviation, which has a ratio of:
$$\sqrt{\frac{n-1}{n}} = \sqrt{\frac{19}{20}} \approx 0.9747$$

The intervals disagree only when $\mu = 0$ lands between the two sets of endpoints, which is when $|\bar{x}|$ falls between the uncorrected and corrected margins. If we evaluated them on separate samples, the differencen of $|\bar{x}|$  between different samples would introduce additional variance (noise), making it much harder to statistically resolve the small difference in their true coverage rates.


### Rerun Part 1 with the number of trials you chose and with 1,000,000. Do your conclusions change about which estimator offers better coverage?

The conclusion stays the same. At both sizes, the corrected interval is consistent with 0.95 and the confidence interval for the uncorrected coverage excludes 0.95.
At 29,196 trials, the uncorrected interval already excludes 0.95. At 1,000,000 trials, the uncorrected interval excludes it by a wider margin, and the interval around the corrected estimate approaches 0.95. The uncorrected interval contains the population mean in only about 94.5% of samples, below the nominal 95%.

In [23]:
# Rerun Part 1 with the number of trials you chose and with 1,000,000.
# Do your conclusions change about which estimator offers better coverage?

n2_reps = 29196
hits_corr_2 = 0
hits_uncorr_2 = 0

for _ in range(n2_reps):
    sample = np.random.normal(0, 1, n)
    xbar = np.mean(sample)
    std_corr = np.std(sample, ddof=1)
    std_uncorr = np.std(sample, ddof=0)
    me_corr = critical_value_t * std_corr / np.sqrt(n)
    me_uncorr = critical_value_t * std_uncorr / np.sqrt(n)
    if xbar - me_corr < 0 and xbar + me_corr > 0:
        hits_corr_2 += 1
    if xbar - me_uncorr < 0 and xbar + me_uncorr > 0:
        hits_uncorr_2 += 1

print(f"Corrected CI contains the population mean in {hits_corr_2} of {n2_reps} repetitions")
print(f"Uncorrected CI contains the population mean in {hits_uncorr_2} of {n2_reps} repetitions")


def report(label, hits, n_reps):
    c_hat = hits / n_reps
    se = np.sqrt(c_hat * (1 - c_hat) / n_reps)
    lo, hi = c_hat - 1.96 * se, c_hat + 1.96 * se
    print(f"{label}: coverage = {c_hat:.4f}, SE = {se:.5f}, 95% CI = ({lo:.4f}, {hi:.4f})")

report("Corrected", hits_corr_2, n2_reps)
report("Uncorrected", hits_uncorr_2, n2_reps)

Corrected CI contains the population mean in 27748 of 29196 repetitions
Uncorrected CI contains the population mean in 27572 of 29196 repetitions
Corrected: coverage = 0.9504, SE = 0.00127, 95% CI = (0.9479, 0.9529)
Uncorrected: coverage = 0.9444, SE = 0.00134, 95% CI = (0.9417, 0.9470)


In [24]:
n3_reps = 1000000
hits_corr_3 = 0
hits_uncorr_3 = 0

for _ in range(n3_reps):
    sample = np.random.normal(0, 1, n)
    xbar = np.mean(sample)
    std_corr = np.std(sample, ddof=1)
    std_uncorr = np.std(sample, ddof=0)
    me_corr = critical_value_t * std_corr / np.sqrt(n)
    me_uncorr = critical_value_t * std_uncorr / np.sqrt(n)
    if xbar - me_corr < 0 and xbar + me_corr > 0:
        hits_corr_3 += 1
    if xbar - me_uncorr < 0 and xbar + me_uncorr > 0:
        hits_uncorr_3 += 1

print(f"Corrected CI contains the population mean in {hits_corr_3} of {n3_reps} repetitions")
print(f"Uncorrected CI contains the population mean in {hits_uncorr_3} of {n3_reps} repetitions")

report("Corrected", hits_corr_3, n3_reps)
report("Uncorrected", hits_uncorr_3, n3_reps)

Corrected CI contains the population mean in 950086 of 1000000 repetitions
Uncorrected CI contains the population mean in 944544 of 1000000 repetitions
Corrected: coverage = 0.9501, SE = 0.00022, 95% CI = (0.9497, 0.9505)
Uncorrected: coverage = 0.9445, SE = 0.00023, 95% CI = (0.9441, 0.9450)


# Part 3

### 1. Imagine you are monitoring the quality of widgets produced at a factory, by sampling 10 widgets every day to compute the mean density of the widgets and see if it matches the expected value. If your monitoring uses 95% confidence intervals, and the widgets are all produced at the expected density, how many days out of the year would you expect your monitoring to raise a false positive alert?

A false positive in mathematics and statistics is an error where classifier incorrectly indicates that a condition is present when it is actually absent. Assuming we are using corrected Standard Deviation (with Bessel's correction) to construct the confidence interval, the coverage is exactly 95% at any n. This means that the false-rate is at 5% per day. For 1 year, the expected value (average) that the monitoring to raise the false positive alert is $$ 0.05 \times 365 = 18.25$$  

3 assumptions is used when constructing this interval:
- The monitoring runs every day of the year (365 days)
- Each day’s sample is independent.
- The sample is normally distributed.

### 2. You discover that the factory’s software does not include Bessel’s correction in its analysis. What effect would this have on the number of days you would expect to see false positives?

Without Bessel’s correction, the SD is divided by $n$ instead of $n−1$, so it’s too small by a factor of $\sqrt(9/10) = 0.9487$ at $n = 10$. The margin of error shrinks by that same factor, which makes the interval about 5% narrower. The false-alert rate rises from about 5.0% to about 6.0% per day, so about 22 days a year instead of 18.25, roughly 3.8 more.

In [38]:
new_n = 10
critical_value_t_new = stats.t.ppf(0.975, df=new_n-1)

n4_reps = 1000000
hits_corr_4 = 0
hits_uncorr_4 = 0

for _ in range(n4_reps):
    sample = np.random.normal(0, 1, new_n)
    xbar = np.mean(sample)
    std_corr = np.std(sample, ddof=1)
    std_uncorr = np.std(sample, ddof=0)
    me_corr = critical_value_t_new * std_corr / np.sqrt(n)
    me_uncorr = critical_value_t_new * std_uncorr / np.sqrt(n)
    if xbar - me_corr < 0 and xbar + me_corr > 0:
        hits_corr_4 += 1
    if xbar - me_uncorr < 0 and xbar + me_uncorr > 0:
        hits_uncorr_4 += 1


miss_uncorr = 1 - hits_uncorr_4 / n4_reps   # false-alert rate per day
expected_days = miss_uncorr * 365
print(expected_days)
print(miss_uncorr)

21.966429999999985
0.06018199999999996


In [31]:
print(1 - hits_corr_4 / n4_reps)

0.04985799999999996


### 3. Now let’s look at monitoring a more productive factory, sampling 10 widgets from the production line every minute for a year. If the factory did not include Bessel’s correction, how many additional false positives would you expect to be raised?

Number of additional false positives in a factory that sample 10 widgets every minute for a year, without including Bessel's correction is about 5470. The assumption is that the monitor runs for 365 days of continuous monitoring and independent samples.

In [33]:
miss_corr   = 1 - hits_corr_4   / n4_reps
miss_uncorr = 1 - hits_uncorr_4 / n4_reps

extra_rate = miss_uncorr - miss_corr          # equivalently (hits_corr_4 - hits_uncorr_4) / n4_reps
print(miss_corr, miss_uncorr, extra_rate)

checks = 365 * 24 * 60
expected_false_positives = extra_rate * checks
print(expected_false_positives)

0.04985799999999996 0.06026600000000004 0.010408000000000084
5470.444800000044


### 4. If the factory switched to sampling 20 widgets every 2 minutes, how many additional false positives would you expect to be raised in a year by neglecting Bessel’s correction?

Number of additional false positives in a factory that sample 20 widgets every 2 minutes for a year, without including Bessel's correction is about 1467. The assumption is that the monitor runs for 365 days of continuous monitoring and independent samples.

In [37]:
n_2 = 20
critical_value_t2 = stats.t.ppf(0.975, df=n_2-1)

n5_reps = 1000000
hits_corr_5 = 0
hits_uncorr_5 = 0

for _ in range(n5_reps):
    sample = np.random.normal(0, 1, n_2)
    xbar = np.mean(sample)
    std_corr = np.std(sample, ddof=1)
    std_uncorr = np.std(sample, ddof=0)
    me_corr = critical_value_t2 * std_corr / np.sqrt(n_2)
    me_uncorr = critical_value_t2 * std_uncorr / np.sqrt(n_2)
    if xbar - me_corr < 0 and xbar + me_corr > 0:
        hits_corr_5 += 1
    if xbar - me_uncorr < 0 and xbar + me_uncorr > 0:
        hits_uncorr_5 += 1

miss_corr_2   = 1 - hits_corr_5   / n5_reps
miss_uncorr_2 = 1 - hits_uncorr_5 / n5_reps   # false-alert rate per day
extra_rate_2 = miss_uncorr_2 - miss_corr_2
checks_2 = 365 * 24 * 30
print(miss_corr_2, miss_uncorr_2, extra_rate_2)
expected_false_positives_2 = extra_rate_2 * checks_2
print(expected_false_positives_2)

0.05019399999999996 0.05577600000000005 0.005582000000000087
1466.9496000000229


### AI reflections:

#### 1. AI prompts used:
- Write me a for loop to repeat the exercise for 10000 times, and show me the vectorized version
- I also had short prompts to use AI for concept explanations, debugging, calculation checks

#### 2. Write a short paragraph (3–5 sentences) addressing: Did the AI produce any errors or oversimplifications? How did you verify the AI’s output? What did you have to correct or add yourself?

For this exercise, I use AI as a as a tutor and code reviewer for this assignment. It explained the concepts behind the questions (especially for part 2 and part 3), and points me to the relevant lecture slide. I verify the AI's output by running the code myself and validate it against the output AI provides. I only have add the explanation more relevant to the lecture than what the AI provided to me.
